In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
# DN = 'C://work/dev/python/progs/texts/sec_bert/'
DN = '/home/jovyan/work/sec_bert/'
os.chdir(DN)

In [ ]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

import matplotlib.pyplot as plt
import seaborn as sns

import os

from ruamel.yaml import YAML
import pandas as pd
import numpy as np
import joblib

from collections import defaultdict
from itertools import chain

import click
import json

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ExponentialLR, MultiStepLR
from torch.utils.data import DataLoader, Dataset


from transformers import BertTokenizer, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModelForMaskedLM, BertConfig, AutoModel
from transformers import DataCollatorWithPadding
from transformers import RobertaTokenizer, RobertaModel

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

import sys
sys.path.append('.')
from src.funcs import set_seed
from src.funcs import metric_multi
from src.funcs import get_opt_thresh, get_preds
from src.funcs import get_conf_df
from src.spec_nn_funcs import TextDFDataset, TextModelClass


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import f1_score

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import RobustScaler

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from ruamel.yaml import YAML

conf = YAML().load(open('params.yaml'))
conf_ttp = YAML().load(open('dvc_pipes/ttp/params_ttp.yaml'))
conf_bert = YAML().load(open('dvc_pipes/bert/params_bert.yaml'))
conf_bert_ttp = YAML().load(open('dvc_pipes/bert_ttp/params_bert_ttp.yaml'))

set_seed(conf['seed'])

In [ ]:
mlb_ttp = joblib.load(conf['prep_text']['ttp_mlb_fn'])
mlb = joblib.load(conf['prep_text']['mlb_fn'])

data_ttp = pd.read_csv(conf_ttp['feat_gen_ttp']['data_fn'])
data_ttp['target'] = data_ttp['target'].map(lambda x: eval(x))
data_ttp['ttp'] = data_ttp['ttp'].map(lambda x: eval(x))
data_ttp['labels'] = data_ttp['labels'].map(lambda x: eval(x))


In [ ]:
data_aug = pd.read_csv('data/temp/aug/data_df.csv')
data_aug['target'] = data_aug['target'].map(lambda x: eval(x))
data_aug['ttp'] = data_aug['ttp'].map(lambda x: eval(x))
data_aug['labels'] = data_aug['labels'].map(lambda x: eval(x))

# Bert

In [ ]:
bert_type = conf_bert['nn_bert']['bert_type']


In [ ]:
VALID_BATCH_SIZE = conf_bert['nn']['batch_size']
TRAIN_BATCH_SIZE = conf_bert['nn']['batch_size']
MAX_SEQ_LENGTH = conf_bert['nn']['maxlen']

checkpoint = 'data/external/models/SecureBERT_Plus/snapshots/4c48ccdb8d2019f179b07dfa27656c655394d78e'
tokenizer = RobertaTokenizer.from_pretrained(checkpoint)
tokenizer_opts = {'max_length':MAX_SEQ_LENGTH, 'return_tensors':"pt", 'padding':True, 'truncation':True, 'add_special_tokens':True}


## ttp

In [ ]:
tr_ttp_ds = TextDFDataset(data_ttp.query('split=="tr"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
val_ttp_ds = TextDFDataset(data_ttp.query('split=="val"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
ts_ttp_ds = TextDFDataset(data_ttp.query('split=="ts"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)

tr_ttp_ld = DataLoader(tr_ttp_ds, batch_size = TRAIN_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
val_ttp_ld = DataLoader(val_ttp_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
ts_ttp_ld = DataLoader(ts_ttp_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))


In [ ]:
name = os.path.basename(conf_bert_ttp['nn_bert_ttp']['model_fn'])
dirname = os.path.dirname(conf_bert_ttp['nn_bert_ttp']['model_fn'])

# model_bert_ttp = torch.load(f'{dirname}/{bert_type}_{name}')
model_bert_ttp = torch.load(conf_bert_ttp['nn_bert_ttp']['model_fn'])

Y_val_proba = np.array(get_preds(model_bert_ttp, ld=val_ttp_ld)['pred'])
Y_tr_proba = np.array(get_preds(model_bert_ttp, ld=tr_ttp_ld)['pred'])
Y_ts_proba = np.array(get_preds(model_bert_ttp, ld=ts_ttp_ld)['pred'])

In [ ]:

thresh_ttp_l = get_opt_thresh(y_true = np.array(data_ttp.loc[data_ttp.split=='tr', 'target'].values.tolist()), 
                          probas = Y_tr_proba, mlb = mlb_ttp, 
                          opt_metric=conf['train_eval_model']['opt_metric'], 
                          thresh_space_l=np.arange(0.001, 1, 0.002), dump_fn = None)

## aug

In [ ]:
tr_aug_ds = TextDFDataset(data_aug.query('split=="tr"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
val_aug_ds = TextDFDataset(data_aug.query('split=="val"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
ts_aug_ds = TextDFDataset(data_aug.query('split=="ts"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)

tr_aug_ld = DataLoader(tr_aug_ds, batch_size = TRAIN_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
val_aug_ld = DataLoader(val_aug_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
ts_aug_ld = DataLoader(ts_aug_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))

In [ ]:
model_bert_aug = torch.load('data/temp/aug/model.pt')

Y_val_aug_proba = np.array(get_preds(model_bert_aug, ld=val_aug_ld)['pred'])
Y_tr_aug_proba = np.array(get_preds(model_bert_aug, ld=tr_aug_ld)['pred'])
Y_ts_aug_proba = np.array(get_preds(model_bert_aug, ld=ts_aug_ld)['pred'])

In [ ]:
thresh_aug_l = get_opt_thresh(y_true = np.array(data_aug.loc[data_aug.split=='tr', 'target'].values.tolist()), 
                          probas = Y_tr_aug_proba, mlb = mlb_ttp, 
                          opt_metric=conf['train_eval_model']['opt_metric'], 
                          thresh_space_l=np.arange(0.001, 1, 0.002), dump_fn = None)

# Качество

## ttp

In [ ]:
ttp_df = pd.concat([data_ttp[['sentence', 'ttp', 'target', 'split']].query('split=="tr"').assign(proba_ttp = Y_tr_proba.tolist()),
          data_ttp[['sentence', 'ttp', 'target', 'split']].query('split=="val"').assign(proba_ttp = Y_val_proba.tolist()),
           data_ttp[['sentence', 'ttp', 'target', 'split']].query('split=="ts"').assign(proba_ttp = Y_ts_proba.tolist())
          ], axis=0, ignore_index=True)

ttp_df['pred_ttp'] = ttp_df['proba_ttp'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_ttp_l)])
ttp_df['pred_str_ttp'] = ttp_df['pred_ttp'].map(lambda x: mlb_ttp.inverse_transform(np.array([x]))[0])

In [ ]:
p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(ttp_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(ttp_df.query('split=="val"')['pred_ttp'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(ttp_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(ttp_df.query('split=="val"')['pred_ttp'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

## aug

In [ ]:
aug_df = pd.concat([data_aug[['sentence', 'ttp', 'target', 'split']].query('split=="tr"').assign(proba_ttp = Y_tr_aug_proba.tolist()),
          data_aug[['sentence', 'ttp', 'target', 'split']].query('split=="val"').assign(proba_ttp = Y_val_aug_proba.tolist()),
           data_aug[['sentence', 'ttp', 'target', 'split']].query('split=="ts"').assign(proba_ttp = Y_ts_aug_proba.tolist())
          ], axis=0, ignore_index=True)

aug_df['pred_ttp'] = aug_df['proba_ttp'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_aug_l)])
aug_df['pred_str_ttp'] = aug_df['pred_ttp'].map(lambda x: mlb_ttp.inverse_transform(np.array([x]))[0])

In [ ]:
p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(aug_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(aug_df.query('split=="val"')['pred_ttp'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(aug_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(aug_df.query('split=="val"')['pred_ttp'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

In [ ]:


ttp_df['enc_ttp'] = ttp_df['ttp'].map(lambda x: mlb_ttp.transform([x])[0])
aug_df['enc_ttp'] = aug_df['ttp'].map(lambda x: mlb_ttp.transform([x])[0])


# Сравним

## синтетика такая же

In [ ]:
data_ttp.query('sentence_source!=sentence').groupby('split').size()

In [ ]:
data_aug.query('sentence_source!=sentence').groupby('split').size()

In [ ]:
data_ttp.groupby('split').size()

In [ ]:
data_aug.groupby('split').size()

In [ ]:
set(data_ttp.query('sentence_source!=sentence')['sentence_source'].unique()) - set(data_aug.query('sentence_source!=sentence')['sentence_source'].unique())

In [ ]:
by_ttp_df = pd.read_csv('data/out/bert_ttp/bert_by_class_metric.csv')
by_aug_df = pd.read_csv('data/temp/aug/bert_by_class_metric.csv')

In [ ]:
diff_classes = by_aug_df.merge(by_ttp_df, on='class').assign(diff=lambda x: abs(x['qual_x']-x['qual_y'])).sort_values(by='diff', ascending=False)['class'].head(10)
by_aug_df.merge(by_ttp_df, on='class').assign(diff=lambda x: abs(x['qual_x']-x['qual_y'])).sort_values(by='diff', ascending=False).head(10)

## T1546.015

In [ ]:
N = 2
ttp = diff_classes.iloc[N]
ttp

In [ ]:
by_aug_df[by_aug_df['class'].map(lambda x: ttp in x)]

In [ ]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
data_aug.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
ttp_df.loc[(ttp_df.split=="val") & (ttp_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'pred_str_ttp']].merge(
    aug_df.loc[(aug_df.split=="val") & (aug_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'pred_str_ttp']], on='sentence', how='outer'
)

### найдем thresh

In [ ]:
ttp_idx = np.where(mlb_ttp.classes_==ttp)[0][0]
aug_idx = np.where(mlb_ttp.classes_==ttp)[0][0]

In [ ]:
thresh_ttp_l[ttp_idx], thresh_aug_l[aug_idx]

### предсказания на валидации


In [ ]:
ttp_df[(ttp_df.ttp.map(lambda x: ttp not in x))&(ttp_df.split=='val')&(ttp_df.pred_str_ttp.map(lambda x: ttp in x))]

In [ ]:
aug_df[(aug_df.ttp.map(lambda x: ttp not in x))&(aug_df.split=='val')&(aug_df.pred_str_ttp.map(lambda x: ttp in x))]

#### f1


##### на валидации

In [ ]:
y_p = ttp_df.assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx])).query('split=="val"')['proba_ttp'].map(lambda x: int(x>=0.119))

y = ttp_df.query('split=="val"')['enc_ttp'].map(lambda y:y[ttp_idx])

f1_score(y, y_p)

In [ ]:
y_p = aug_df.assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx])).query('split=="val"')['proba_ttp'].map(lambda x: int(x>=0.305))

y = aug_df.query('split=="val"')['enc_ttp'].map(lambda y:y[ttp_idx])

f1_score(y, y_p)

##### на трейне

In [ ]:
y_p = ttp_df.assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx])).query('split=="tr"')['proba_ttp'].map(lambda x: int(x>=0.119))

y = ttp_df.query('split=="tr"')['enc_ttp'].map(lambda y:y[ttp_idx])

f1_score(y, y_p)

In [ ]:
from sklearn.metrics import precision_score, recall_score

In [ ]:
y_p = aug_df.assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx])).query('split=="tr"')['proba_ttp'].map(lambda x: int(x>=0.119))

y = aug_df.query('split=="tr"')['enc_ttp'].map(lambda y:y[ttp_idx])

f1_score(y, y_p), precision_score(y, y_p), recall_score(y, y_p)

In [ ]:
y_p1 = np.where(y_p==1)[0]
y_p1

In [ ]:
fp_idx = set(y_p1) - set(np.where(y==1)[0])
fp_idx

In [ ]:
y_p = aug_df.assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx])).query('split=="tr"')['proba_ttp'].map(lambda x: int(x>=0.305))

y = aug_df.query('split=="tr"')['enc_ttp'].map(lambda y:y[ttp_idx])

f1_score(y, y_p), precision_score(y, y_p), recall_score(y, y_p)

In [ ]:
y_p1 = np.where(y_p==1)[0]
y_p1

In [ ]:
fp_idx = set(y_p1) - set(np.where(y==1)[0])
fp_idx

Видим, что подрос precision сильно, recall упал, TP/(TP+FP), ибо увеличился TP, ибо уменьшился FP. Среди предсказанных как 1 ищем проблемные

In [ ]:
aug_df.assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[aug_idx])).query('split=="tr"').iloc[[450, 534, 536, 539, 540, 545, 1062, 1280, 7328, 24103]]

## T1555.004


Крошечный класс, и то, угадываются подклассы, видимо, просто обучение из-за разницы в выборках по другому пошло, с другими пакетами, так как датасеты разные. Отличия несущественные


In [ ]:
N = 0
ttp = diff_classes.iloc[N]
ttp

In [ ]:
by_aug_df[by_aug_df['class'].map(lambda x: ttp in x)]

In [ ]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
data_aug.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
ttp_df.loc[(ttp_df.split=="val") & (ttp_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'pred_str_ttp']].merge(
    aug_df.loc[(aug_df.split=="val") & (aug_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'pred_str_ttp']], on='sentence', how='outer'
)

In [ ]:
data = pd.read_csv(conf['prep_text']['prep_fn'])


data['ttp'] = data['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20


sel_dop = (data.split=='tr')
mini_ttp_l = data[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

In [ ]:
data_s= pd.read_csv('data/temp/aug/prep_df.csv')

data_s['ttp'] = data_s['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20

sel_dop = (data_s.split=='tr')
mini_ttp_l = data_s[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

### найдем thresh

In [ ]:
ttp_idx = np.where(mlb_ttp.classes_==ttp)[0][0]
aug_idx = np.where(mlb_ttp.classes_==ttp)[0][0]

In [ ]:
thresh_ttp_l[ttp_idx], thresh_aug_l[aug_idx]

In [ ]:
opt_metric_aug_df = pd.read_csv('data/temp/aug/bert_opt_metric.csv').set_index('Unnamed: 0').loc[lambda x:x['class_nm']==ttp]
idx_aug = opt_metric_aug_df['f1'].argmax()
idx_aug

In [ ]:
opt_metric_df = pd.read_csv(conf_bert_ttp['nn_bert_ttp']['opt_metric_fn']).set_index('Unnamed: 0').loc[lambda x:x['class_nm']==ttp]

idx = opt_metric_df['f1'].argmax()
idx

### предсказания на валидации


In [ ]:
ttp_df[(ttp_df.ttp.map(lambda x: ttp not in x))&(ttp_df.split=='val')&(ttp_df.pred_str_ttp.map(lambda x: ttp in x))]

In [ ]:
aug_df[(aug_df.ttp.map(lambda x: ttp not in x))&(aug_df.split=='val')&(aug_df.pred_str_ttp.map(lambda x: ttp in x))]

#### f1


##### на валидации

In [ ]:
y_p = ttp_df.assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx])).query('split=="val"')['proba_ttp'].map(lambda x: int(x>=0.197))

y = ttp_df.query('split=="val"')['enc_ttp'].map(lambda y:y[ttp_idx])

f1_score(y, y_p)

In [ ]:
y_p = aug_df.assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx])).query('split=="val"')['proba_ttp'].map(lambda x: int(x>=0.527))

y = aug_df.query('split=="val"')['enc_ttp'].map(lambda y:y[ttp_idx])

f1_score(y, y_p)

##### на трейне

In [ ]:
y_p = ttp_df.assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx])).query('split=="tr"')['proba_ttp'].map(lambda x: int(x>=0.197))

y = ttp_df.query('split=="tr"')['enc_ttp'].map(lambda y:y[ttp_idx])

f1_score(y, y_p)

In [ ]:
from sklearn.metrics import precision_score, recall_score

In [ ]:
y_p = aug_df.assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx])).query('split=="tr"')['proba_ttp'].map(lambda x: int(x>=0.197))

y = aug_df.query('split=="tr"')['enc_ttp'].map(lambda y:y[ttp_idx])

f1_score(y, y_p), precision_score(y, y_p), recall_score(y, y_p)

In [ ]:
y_p1 = np.where(y_p==1)[0]
y_p1

In [ ]:
fp_idx = set(y_p1) - set(np.where(y==1)[0])
fp_idx

In [ ]:
y_p = aug_df.assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx])).query('split=="tr"')['proba_ttp'].map(lambda x: int(x>=0.527))

y = aug_df.query('split=="tr"')['enc_ttp'].map(lambda y:y[ttp_idx])

f1_score(y, y_p), precision_score(y, y_p), recall_score(y, y_p)

Видим, что подрос precision, recall не поменялся, TP/(TP+FP), ибо увеличился TP, ибо уменьшился FP. Среди предсказанных как 1 ищем проблемные

In [ ]:
aug_df.assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[aug_idx])).query('split=="tr"').iloc[[977, 5436, 5440, 15975, 25351]]

In [ ]:
comp_df = ttp_df.loc[(ttp_df.ttp.map(lambda x: ttp in x))&(ttp_df.split=="tr"), ['sentence','proba_ttp', 'pred_str_ttp', 'ttp']]\
        .assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx]))\
    .merge(
aug_df.loc[(aug_df.ttp.map(lambda x: ttp in x))&(aug_df.split=="tr"), ['sentence','proba_ttp', 'pred_str_ttp']]\
        .assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[aug_idx])), on='sentence', how='outer'
        
    )

comp_df

In [ ]:
comp_df['proba_ttp_x'].describe(), comp_df['proba_ttp_y'].describe()

## T1546.015

Тут cut-off меняется туда-сюда

In [ ]:
N = 7
ttp = diff_classes.iloc[N]
ttp

In [ ]:
by_aug_df[by_aug_df['class'].map(lambda x: ttp in x)]

In [ ]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
data_aug.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

In [ ]:
ttp_df.loc[(ttp_df.split=="val") & (ttp_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'pred_str_ttp']].merge(
    aug_df.loc[(aug_df.split=="val") & (aug_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'pred_str_ttp']], on='sentence', how='outer'
)

In [ ]:
data = pd.read_csv(conf['prep_text']['prep_fn'])


data['ttp'] = data['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20


sel_dop = (data.split=='tr')
mini_ttp_l = data[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

In [ ]:
data_s= pd.read_csv('data/temp/aug/prep_df.csv')

data_s['ttp'] = data_s['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20

sel_dop = (data_s.split=='tr')
mini_ttp_l = data_s[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

## найдем thresh

In [ ]:
ttp_idx = np.where(mlb_ttp.classes_==ttp)[0][0]
aug_idx = np.where(mlb_ttp.classes_==ttp)[0][0]

In [ ]:
thresh_ttp_l[ttp_idx], thresh_aug_l[aug_idx]

<div> Кажется, будто у нас ложноположительных больше стало на валидации
</div>

In [ ]:
opt_metric_aug_df = pd.read_csv('data/temp/aug/bert_opt_metric.csv').set_index('Unnamed: 0').loc[lambda x:x['class_nm']==ttp]
idx_aug = opt_metric_aug_df['f1'].argmax()
idx_aug

In [ ]:
opt_metric_aug_df.iloc[idx_aug-2:idx_aug+20]

In [ ]:
opt_metric_df = pd.read_csv(conf_bert_ttp['nn_bert_ttp']['opt_metric_fn']).set_index('Unnamed: 0').loc[lambda x:x['class_nm']==ttp]

idx = opt_metric_df['f1'].argmax()
idx

In [ ]:
ttp_df[(ttp_df.ttp.map(lambda x: ttp not in x))&(ttp_df.pred_str_ttp.map(lambda x: ttp in x))]

In [ ]:
aug_df[(aug_df.ttp.map(lambda x: ttp not in x))&(aug_df.pred_str_ttp.map(lambda x: ttp in x))]

In [ ]:
comp_df = ttp_df.loc[(ttp_df.ttp.map(lambda x: ttp in x))&(ttp_df.split=="tr"), ['sentence','proba_ttp', 'pred_str_ttp']]\
        .assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx]))\
    .merge(
aug_df.loc[(aug_df.ttp.map(lambda x: ttp in x))&(aug_df.split=="tr"), ['sentence','proba_ttp', 'pred_str_ttp']]\
        .assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[aug_idx])), on='sentence', how='outer'
        
    )

comp_df

In [ ]:
comp_df['proba_ttp_x'].describe(), comp_df['proba_ttp_y'].describe()

- То есть, чтобы поймать некоторые предложения во втором случае пришлось снизить планку, так как в целом менее уверена такая модель

In [ ]:
with pd.option_context('display.max_colwidth', 100):
    display(comp_df.assign(dif = lambda x: x['proba_ttp_y']-x['proba_ttp_x']).sort_values(by='dif')['sentence'].iloc[:5])

In [ ]:
s = 'KONNI has modified ComSysApp service to load the malicious DLL payload'

In [ ]:
conf['feat_eng']['feat_final_fn']

In [ ]:
from src.interpret.close_k_feat import select_k_neigh
feat_data = pd.read_csv('data/temp/aug/feat_final_df.csv')

In [ ]:
idx = data_aug[data_aug.sentence.str.contains(s)].index[0]
data_aug.index.get_loc(idx)

In [ ]:
comp_df1, select_df1 = select_k_neigh(data=data_aug, feat_data=feat_data, k=10, idx=[14489])


with pd.option_context('display.max_colwidth', 200):
    display(comp_df1)

## T1102

In [ ]:
ttp = 'T1102'

In [ ]:
aug_df[(aug_df['ttp'].map(lambda x: ttp in x))&(aug_df.split=='val')]

In [ ]:
ttp_df[(ttp_df['ttp'].map(lambda x: ttp in x))&(ttp_df.split=='val')]

## T1572

In [ ]:
ttp = 'T1572'

In [ ]:
aug_df[(aug_df['ttp'].map(lambda x: ttp in x))&(aug_df.split=='val')]

In [ ]:
ttp_df[(ttp_df['ttp'].map(lambda x: ttp in x))&(ttp_df.split=='val')]

# Генерим

## 1102

In [ ]:
# путаница с T1608.001, который очень близкий
# https://attack.mitre.org/techniques/T1608/001/
# https://attack.mitre.org/techniques/T1102/
ttp_df[(ttp_df['ttp'].map(lambda x: 'T1102' in x)) & (ttp_df.split=='val')]

## 1572

In [ ]:
# путаница с T1608.001, который очень близкий
# https://attack.mitre.org/techniques/T1608/001/
# https://attack.mitre.org/techniques/T1102/
ttp_df[(ttp_df['ttp'].map(lambda x: 'T1572' in x)) & (ttp_df.split=='val')]

## генерация для списка

In [ ]:
data = pd.read_csv(conf['prep_text']['prep_fn'])
data['ttp'] = data['ttp'].map(lambda x: eval(x))
data.shape

In [ ]:
Сначала сэмплировал 2 класса:
- с аугментацией: 
    - f1 micro - 0.55 
    - f1 macro - 0.453 (T1102 - 0.33, T1572 - 0.3, T1555 - 0.3, T1059 - 0)
    - pr_auc - 0.511
- без:
    - f1 micro - 0.54  
    - f1_macro - 0.453 (T1102 - 0.53, T1572- 0, T1555 - 0, T1059 - 0)
    - pr_auc - 0.516


In [ ]:
Во второй раз, поменял сид и уменьшил общий слой берта:
- с аугментацией: 
    - f1 micro - 0.55 
    - f1 macro - 0.446 (T1102 - 0.25, T1572 - 0.36, T1059 - 0, T1555- 0)
    - pr_auc - 0.511
- без:
    - f1 micro - 0.55  
    - f1_macro - 0.458 (T1102 - 0.35, T1572- 0.44, T1059 - 0, T1555 - 0.2)
    - pr_auc - 0.49




Коля, привет! Я провел эксперимент с аугментацией (добавил по 50 экземпляров для классов 1102, 1572, T1059, T1555), в целом сдвигов по качеству нет, изменяется на 2% туда-сюда. То лучше, то хуже в зависимости от случайного инициализатора. Чаще качество на сэмлированных классах повышалось, однако эффект компенсируется плавающим качеством по маленьким классам (у которых около 15-20 примеров), для них угадывание на 1 больше - TP или лишний False Positive смещает сильно f1. Пробовал менять количество нейроновой в общем слое берта, чтобы снизить возможности для переобучения, но особо не поменяло ситуацию. Не вижу смысла сэмплировать больше.
 
Ниже статистика:
Сначала сэмплировал 2 класса:
- с аугментацией: 
    - f1 micro - 0.55 
    - f1 macro - 0.453 (T1102 - 0.33, T1572 - 0.3, T1555 - 0.3, T1059 - 0)
    - pr_auc - 0.511
- без:
    - f1 micro - 0.54  
    - f1_macro - 0.453 (T1102 - 0.53, T1572- 0, T1555 - 0, T1059 - 0)
    - pr_auc - 0.516
Во второй раз, поменял сид и уменьшил общий слой берта:
- с аугментацией: 
    - f1 micro - 0.55 
    - f1 macro - 0.446 (T1102 - 0.25, T1572 - 0.36, T1059 - 0, T1555- 0)
    - pr_auc - 0.511
- без:
    - f1 micro - 0.55  
    - f1_macro - 0.458 (T1102 - 0.35, T1572- 0.44, T1059 - 0, T1555 - 0.2)
    - pr_auc - 0.49



In [ ]:
N = 10
repeate_num = 5

ttp_l = ['T1059', 'T1555']

DN = 'data/external/chatgpt'

def save_synth_gpt_ttp(data, ttp_nm, n_samples, repeate_num, save_dn):
    res_l = []
    
    sample_df = data[(data['ttp'].map(lambda x: ttp_nm in x)) & (data.split=='tr')].sample(n=n_samples)    
    for _, row in sample_df.iterrows():
        sents = create_rephrase(row['sentence'], num_variants=repeate_num)
        res_l.extend([row.copy().to_frame().T.assign(url=lambda x: f'synth\n{x["url"].iloc[0]}\n{x["sentence"].iloc[0]}', sentence=sent) for sent in sents])
        
    synth_df = pd.concat(res_l, ignore_index=True)   
    synth_df.to_csv(f'{save_dn}/{ttp_nm}.csv', index=False)

for ttp in ttp_l:
    save_synth_gpt_ttp(data, ttp_nm=ttp, n_samples=N, repeate_num=repeate_num, save_dn=DN)





## считывание

In [ ]:
synth_df = pd.concat([pd.read_csv(f'{DN}/{it}') for it in os.listdir(DN) if not '.ipynb_checkpoints' in it], ignore_index=True)

pd.concat([data, synth_df], ignore_index=True)

## функции синтетики

In [ ]:
def create_rephrase(sent, num_variants=5):
    return [sent[:10]]*num_variants

In [ ]:
- расширение датасета
    - 100, 1
    - для пары классов 50 нагенерю значений

- предложены 
    - для ттп сильно ошибается, парочка которая хорошо определилась, как ведет при плохих и хороших вариках.
